Robotics and Embodied AI: Diffusion Policy, ACT, and Action Chunking

Imitation learning paradigms for continuous robotic manipulation and visuomotor control:
1. Action Chunking with Ensembling (ACT): Trajectory-level prediction and smooth temporal ensembling.
2. Diffusion Policy: Reverse DDPM score-based denoising over continuous multi-joint action spaces.
3. Conditional VAE Latent Regularization: Analytical KL divergence over demonstrator execution styles.
4. Robot Control Safety Envelope: Velocity and coordinate clamping under physical actuator bounds.

## 2. Mathematical Formulations & Theoretical Foundations

- **ACT Temporal Ensembling**:
  $$w_k = \exp(-m \cdot k), \quad \bar{a}_t = \frac{\sum_{k=0}^{K-1} w_k a_{t, k}}{\sum_{k=0}^{K-1} w_k}$$
- **DDPM Reverse Action Denoising Step**:
  $$a_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( a_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(a_t, s, t) \right) + \sigma_t z$$
- **ACT CVAE Latent KL Loss**:
  $$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^{D_z} \left( 1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2 \right)$$
- **Actuator Safety Clamping**:
  $$a_{\text{safe}} = \operatorname{clamp}(a, a_{\min}, a_{\max})$$

In [ ]:
import math
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "13_robotics_embodied_ai_diffusion_policy_and_act.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. Action Chunking with Ensembling (ACT)
def temporal_ensemble_actions(action_preds: torch.Tensor, decay: float = 0.1) -> torch.Tensor:
    k = action_preds.shape[1]
    steps = torch.arange(k, dtype=action_preds.dtype, device=action_preds.device)
    weights = torch.exp(-decay * steps)
    weights = weights / torch.sum(weights)
    return torch.sum(action_preds * weights.view(1, -1, 1), dim=1)

ens_calc = temporal_ensemble_actions(golden["action_preds"], decay=0.1)
diff_ens = torch.max(torch.abs(ens_calc - golden["ensembled_ref"])).item()
print(f"ACT Temporal Ensemble max abs error: {diff_ens:.2e}")
assert diff_ens < 1e-5

In [ ]:
# 2. Diffusion Policy Reverse DDPM Step
def diffusion_policy_ddpm_step(
    action: torch.Tensor,
    noise_pred: torch.Tensor,
    noise_z: torch.Tensor,
    alpha_t: float,
    alpha_bar_t: float,
    sigma_t: float,
) -> torch.Tensor:
    coeff = (1.0 - alpha_t) / ((1.0 - alpha_bar_t) ** 0.5)
    mu = (1.0 / (alpha_t ** 0.5)) * (action - coeff * noise_pred)
    return mu + sigma_t * noise_z

denoised_calc = diffusion_policy_ddpm_step(
    golden["action_t"],
    golden["noise_pred"],
    golden["noise_z"],
    alpha_t=0.99,
    alpha_bar_t=0.85,
    sigma_t=0.05,
)
diff_denoised = torch.max(torch.abs(denoised_calc - golden["denoised_ref"])).item()
print(f"Diffusion Policy DDPM Step max abs error: {diff_denoised:.2e}")
assert diff_denoised < 1e-5

In [ ]:
# 3. ACT CVAE KL Loss
def act_cvae_kl_loss(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    kl = -0.5 * torch.sum(1.0 + logvar - mu ** 2 - torch.exp(logvar), dim=-1)
    return kl.mean()

kl_calc = act_cvae_kl_loss(golden["cvae_mu"], golden["cvae_logvar"])
diff_kl = torch.abs(kl_calc - golden["kl_loss_ref"]).item()
print(f"ACT CVAE KL Loss max abs error: {diff_kl:.2e}")
assert diff_kl < 1e-5

In [ ]:
# 4. Robot Action Safety Clamp & Summary Table
def robot_action_safety_clamp(actions: torch.Tensor, a_min: torch.Tensor, a_max: torch.Tensor) -> torch.Tensor:
    return torch.clamp(actions, min=a_min.view(1, 1, -1), max=a_max.view(1, 1, -1))

clamped_calc = robot_action_safety_clamp(golden["raw_actions"], golden["a_min"], golden["a_max"])
diff_clamp = torch.max(torch.abs(clamped_calc - golden["clamped_ref"])).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'ACT Temporal Ensemble':<32} | {diff_ens:<15.2e} | PASSED")
print(f"{'Diffusion Policy DDPM Step':<32} | {diff_denoised:<15.2e} | PASSED")
print(f"{'ACT CVAE KL Regularization':<32} | {diff_kl:<15.2e} | PASSED")
print(f"{'Robot Safety Clamping':<32} | {diff_clamp:<15.2e} | PASSED")
print("=" * 65)
assert diff_clamp < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] ACT trajectory chunking circumvents single-step compounding errors and temporal discontinuities.
- [x] Temporal ensembling exponentially filters overlapping query action chunks for smooth mechanical execution.
- [x] Diffusion Policy generates non-Gaussian multimodal action trajectories without mode collapse.
- [x] Actuator safety envelopes guarantee commanded velocities remain within physical motor boundaries.